In [1]:
!pip -q install insightface onnxruntime-gpu==1.20.2 batch-face opencv-python-headless pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.5/291.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.6/30.6 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.1 MB/s eta 0:00:00


In [2]:
import os
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = sec.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = sec.get_secret('KAGGLE_KEY')
!kaggle datasets download -d shahriarsamee/vehicle-detection-full -p /kaggle/working/data --unzip
!echo "--- disk ---" && df -h /kaggle/working

Dataset URL: https://www.kaggle.com/datasets/shahriarsamee/vehicle-detection-full
License(s): unknown
100%|██████████████████████████████████████| 4.49G/4.49G [03:47<00:00, 21.2MB/s]

--- disk ---
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G  4.6G   15G  24% /kaggle/working


In [3]:
import glob, os, shutil, yaml

code = glob.glob('/kaggle/input/**/run_all.py', recursive=True)
assert code, "run_all.py not found"
CODE_DIR = '/kaggle/working/face_anonymization'
shutil.copytree(os.path.dirname(code[0]), CODE_DIR, dirs_exist_ok=True)

DATA = '/kaggle/working/data/Vehicle_Detection_Dataset'
assert os.path.isdir(DATA), f"dataset missing at {DATA}"

cfg_path = os.path.join(CODE_DIR, 'config.yaml')
cfg = yaml.safe_load(open(cfg_path))
cfg['structure']['variants'] = ["merged(all)_images_dataset"]
yaml.safe_dump(cfg, open(cfg_path, 'w'))

chk = yaml.safe_load(open(cfg_path))
assert chk['structure']['variants'] == ["merged(all)_images_dataset"], f"lock failed: {chk['structure']['variants']}"
print("Locked to:", chk['structure']['variants'])

Locked to: ['merged(all)_images_dataset']


In [4]:
DATA = '/kaggle/working/data/Vehicle_Detection_Dataset'
!cd /kaggle/working/face_anonymization && python run_all.py \
  --dataset-root "{DATA}" \
  --output-root "/kaggle/working/output" \
  --work-dir "/kaggle/working/work" \
  --device cuda
!echo "--- disk after run ---" && df -h /kaggle/working 

effective config written to /kaggle/working/work/_effective_config.yaml

>>> 00_env_check.py 
=== library versions ===
numpy 2.1.3 | opencv 4.14.0
onnxruntime 1.20.2 providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
torch 2.11.0+cu128 cuda? True

=== GPU summary ===
GPU READY for BOTH detectors (SCRFD via onnxruntime-gpu, RetinaFace via torch).

=== detector weight download / load ===
download_path: /root/.insightface/models/buffalo_l
100%|████████████████████████████████| 281857/281857 [00:03<00:00, 88616.29KB/s]
Downloading: "https://github.com/elliottzheng/face-detection/releases/download/0.0.1/Resnet50_Final.pth" to /root/.cache/torch/hub/checkpoints/Resnet50_Final.pth
100%|████████████████████████████████████████| 104M/104M [00:02<00:00, 46.2MB/s]
SCRFD loaded OK; RetinaFace loaded OK.

=== dataset paths ===
dataset_root: /kaggle/working/data/Vehicle_Detection_Dataset EXISTS
found at least 5000 images under the configured structure (full cou

In [5]:
import glob, os, zipfile
zips = glob.glob('/kaggle/working/**/*.zip', recursive=True)
if not zips: print("WARNING: no zip produced — check run log above")
for z in zips:
    gb = os.path.getsize(z)/1e9
    try:
        with zipfile.ZipFile(z) as f:
            ok = f.testzip() is None; n = len(f.namelist())
        print(f"{z}\n  size={gb:.2f} GB  entries={n}  integrity_OK={ok}")
    except Exception as e:
        print(f"{z}\n  COULD NOT OPEN: {e}")
print("\n^ need integrity_OK=True before trusting this zip")

/kaggle/working/anonymized_dataset.zip
  size=4.29 GB  entries=13990  integrity_OK=True
/kaggle/working/anonymization_reports.zip
  size=0.00 GB  entries=12  integrity_OK=True

^ need integrity_OK=True before trusting this zip
